# Weekly U.S. Natural Gas Storage Forecast
Forecasts the weekly change in Lower 48 working gas storage (EIA, released Thursdays) from heating and cooling degree days.

## 1. Load storage data
Source: https://www.eia.gov/dnav/ng/hist/nw2_epg0_swo_r48_bcfw.htm

In [ ]:
import pandas as pd
import requests
import statsmodels.formula.api as smf

df = pd.read_csv("storage.csv", header=None, names=["date", "storage_bcf"])
df = df.dropna(how="all")
df.head()


Convert dates and sort from the oldest week to the newest.

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values(by="date").reset_index(drop=True)
df.tail()

Weekly change = this week's storage minus last week's. Positive is an injection, negative a withdrawal.

In [ ]:
df["weekly_change"] = df["storage_bcf"].diff()
df.head()

Seasonal cycle: injections in summer, withdrawals in winter.

In [ ]:
df.plot(x="date", y="weekly_change");

## 2. Weather data
Daily mean temperature from the Open-Meteo archive. HDD = max(65°F - temp, 0), CDD = max(temp - 65°F, 0), summed into Friday-ending weeks to match EIA.

In [ ]:
url = "https://archive-api.open-meteo.com/v1/archive"

def get_weekly_weather(lat, lon):
  params = {
      "latitude": lat,
      "longitude": lon,
      "start_date": "2010-01-01",
      "end_date": "2026-10-30",
      "daily": "temperature_2m_mean",
      "temperature_unit": "fahrenheit",
      "timezone": "America/Chicago"
  }
  response = requests.get(url, params=params)
  data = response.json()
  city = pd.DataFrame(data["daily"])
  city["time"] = pd.to_datetime(city["time"])
  city["hdd"] = (65 - city["temperature_2m_mean"]).clip(lower=0)
  city["cdd"] = (city["temperature_2m_mean"]-65).clip(lower=0)
  return city.resample("W-FRI", on="time")[["hdd","cdd"]].sum()

HDD cities: Midwest and Northeast heating demand. CDD cities: Texas power burn for air conditioning.

In [ ]:
cities = {
    "chicago": (41.88, -87.63),
    "detroit": (42.33, -83.05),
    "minneapolis": (44.98, -93.27),
    "new_york": (40.71, -74.01),
    "philadelphia": (39.95, -75.17),
    "boston": (42.36, -71.06),
    "houston": (29.76, -95.37),
    "dallas": (32.78, -96.80)
}

In [ ]:
all_weather = {}
for name, (lat, lon) in cities.items():
  all_weather[name] = get_weekly_weather(lat, lon)
  print(name, "done")

In [ ]:
hdd_cities = ["chicago", "detroit", "minneapolis","new_york", "philadelphia", "boston"]
cdd_cities = ["houston", "dallas"]

Average HDD across the northern cities and CDD across the Texas cities.

In [ ]:
combined = pd.DataFrame({
    "hdd": pd.concat([all_weather[c]["hdd"] for c in hdd_cities], axis=1).mean(axis=1),
    "cdd": pd.concat([all_weather[c]["cdd"] for c in cdd_cities], axis=1).mean(axis=1)
})
combined.tail()

## 3. Full-sample regression
OLS: weekly_change ~ hdd + cdd, fit on all weeks (in-sample).

In [ ]:
weekly2 = df.merge(combined, left_on="date", right_index=True, how="inner").dropna()

model2 = smf.ols("weekly_change ~ hdd + cdd", data=weekly2).fit()
print(model2.rsquared)

weekly2["error"] = weekly2["weekly_change"] - model2.predict(weekly2)
print(weekly2["error"].abs().mean())

## 4. Out-of-sample test
Train on 2010-2022, test on 2023 onward. Naive benchmark: same week last year (`shift(52)`).

In [ ]:
weekly2["last_year"] = weekly2["weekly_change"].shift(52)
train = weekly2[weekly2["date"] < "2023-01-01"].copy()
test = weekly2[weekly2["date"] >= "2023-01-01"].copy()
print(len(train), len(test))

In [ ]:
model3 = smf.ols("weekly_change ~ hdd + cdd", data=train).fit()
print(model3.summary())

Model error on the test weeks (mean absolute error, Bcf).

In [ ]:
test["error"] = test["weekly_change"] - model3.predict(test)
print(test["error"].abs().mean())

Naive benchmark error on the same test weeks.

In [ ]:
test["naive_error"] = test["weekly_change"] - test["last_year"]
print(test["naive_error"].abs().mean())

## 5. Actual vs. model (test period)

In [ ]:
ax = test.plot(x="date", y="weekly_change", label="Actual")
test.assign(predicted=model3.predict(test)).plot(x="date", y="predicted", label="Model", ax=ax,
     title="Weekly storage change: actual vs. model (out-of-sample, 2023-2026)", ylabel="Bcf");


In [ ]:
ax.get_figure().savefig("actual_vs_model.png", dpi=150, bbox_inches="tight")